# 12 — Полный цикл CrafText с real Qwen/Tunix actor: rollout → replay → score → PPO update

Эта тетрадка связывает synchronous CrafText pipeline в один обучающий цикл без offline/mock backend: batched CrafText rollout, MegaPrompts, настоящий `QwenTunixBackend`, сохранение replay evidence, преобразование replay в `TextTrajectoryBatch`, real Qwen actor scoring через `TunixCausalLmActor` и один full-token trainable PPO smoke update.

Тетрадка **не скачивает веса**. Перед запуском положите approved local snapshot в `artifacts/models/qwen25-05b-instruct` и установите extras:

```bash
pyenv exec python -m uv sync --extra envs --extra prompts --extra tunix --extra examples
```

Интерактивные размеры ограничены переменными `TCX_NOTEBOOK_BATCH_SIZE` и `TCX_NOTEBOOK_HORIZON`; для benchmark используйте scripts/perf lane, а не Jupyter execution.


In [ ]:
from pathlib import Path
import os
import subprocess

import jax
import jax.numpy as jnp

from tunix_craftext.algorithms import masked_token_returns
from tunix_craftext.batched_rollout import collect_batched_text_rollout, replays_from_batched_rollout
from tunix_craftext.config import load_mvp_config
from tunix_craftext.learner import create_token_state, full_token_ppo_update, token_actor_critic_outputs, token_ppo_update
from tunix_craftext.profiling import PhaseProfiler, block_until_ready, save_profile
from tunix_craftext.prompts import MegaPromptRenderer
from tunix_craftext.replay import save_replay
from tunix_craftext.runtime import build_craftext_runtime
from tunix_craftext.text_trajectory import text_trajectory_from_replay
from tunix_craftext.tunix_actor import build_qwen_tunix_actor, init_linear_value_head
from tunix_craftext.tunix_adapter import QwenTunixBackend


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook inside tunix-craftext.')

SNAPSHOT = ROOT / 'artifacts' / 'models' / 'qwen25-05b-instruct'
if not SNAPSHOT.is_dir():
    raise FileNotFoundError(
        f'Expected approved local Qwen snapshot at {SNAPSHOT}; no download is implicit.'
    )

config_path = ROOT / 'configs' / 'mvp' / 'qwen_craftext.yaml'
model_profile_path = ROOT / 'configs' / 'models' / 'qwen25_05b_instruction.yaml'
config = load_mvp_config(config_path)
runtime = build_craftext_runtime(config)

BATCH_SIZE = int(os.environ.get('TCX_NOTEBOOK_BATCH_SIZE', min(config.environment.batch_size, 2)))
HORIZON = int(os.environ.get('TCX_NOTEBOOK_HORIZON', min(config.environment.horizon, 4)))
MAX_NEW_TOKENS = int(os.environ.get('TCX_NOTEBOOK_MAX_NEW_TOKENS', 8))
CACHE_SIZE = int(os.environ.get('TCX_QWEN_CACHE_SIZE', 2048))
QWEN_HIDDEN_DIM = 896

profiler = PhaseProfiler(enable_nvtx=bool(int(os.environ.get('TCX_ENABLE_NVTX', '0'))))

def git_revision() -> str:
    try:
        return subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
    except (OSError, subprocess.CalledProcessError):
        return 'unversioned'

print('CrafText actions:', runtime.actions.labels)
print('Qwen snapshot:', SNAPSHOT)
print('interactive B/T/max_new_tokens:', BATCH_SIZE, HORIZON, MAX_NEW_TOKENS)


In [ ]:
renderer = MegaPromptRenderer(config.prompt.template)
backend = QwenTunixBackend(SNAPSHOT, cache_size=CACHE_SIZE, seed=config.run.seed)
value_head = init_linear_value_head(jax.random.PRNGKey(config.run.seed), hidden_dim=QWEN_HIDDEN_DIM)
actor = build_qwen_tunix_actor(
    profile_path=model_profile_path,
    snapshot=SNAPSHOT,
    cache_size=CACHE_SIZE,
    value_head=value_head,
    seed=config.run.seed,
)
fallback_action_id = runtime.actions.index_of('NOOP')
print('actor profile:', actor.profile.model_id)


In [ ]:
with profiler.section('rollout'):
    rollout = collect_batched_text_rollout(
        runtime.adapter,
        renderer,
        backend,
        actions=runtime.actions,
        batch_size=BATCH_SIZE,
        horizon=HORIZON,
        seed=config.run.seed,
        goal='Collect useful CrafText experience for a tiny PPO update.',
        max_new_tokens=MAX_NEW_TOKENS,
        invalid_action='fallback',
        fallback_action_id=fallback_action_id,
    )

for t, decision in enumerate(rollout.decisions):
    print('t=', t, 'actions=', [action.label for action in decision.actions], 'reward=', decision.transition.reward.tolist())


In [ ]:
print('JAX backend:', jax.default_backend())
print('visible devices:', jax.devices())


In [ ]:
output_dir = ROOT / 'artifacts' / 'trajectories' / 'full-cycle-craftext-training'
with profiler.section('replay'):
    replays = replays_from_batched_rollout(
        rollout,
        config_path=str(config_path.relative_to(ROOT)),
        commit=git_revision(),
        backend='tunix-single-device:Qwen/Qwen2.5-0.5B-Instruct',
    )
    for env_index, replay in enumerate(replays):
        save_replay(output_dir / f'env-{env_index}.json', replay)
        print('saved replay', env_index, 'steps=', len(replay.steps))


In [ ]:
with profiler.section('trajectory'):
    batches = tuple(text_trajectory_from_replay(replay) for replay in replays)
    batch = batches[0]

with profiler.section('actor'):
    qwen_scores = actor.score_tokens(
        prompt_token_ids=batch.prompt_token_ids,
        prompt_token_mask=batch.prompt_token_mask,
        token_ids=batch.token_ids,
        token_mask=batch.token_mask,
    )
    block_until_ready(qwen_scores.token_logprobs)

returns = masked_token_returns(batch.rewards, batch.token_mask, gamma=0.99)
old_actor_state = create_token_state(
    jax.random.PRNGKey(42),
    token_bucket_count=512,
    hidden=64,
    learning_rate=3e-4,
)
old_new_logprobs, old_values, old_entropy = token_actor_critic_outputs(old_actor_state, batch)
with profiler.section('ppo'):
    actor_state, metrics = full_token_ppo_update(old_actor_state, batch, gamma=0.99)
    block_until_ready(metrics)
new_new_logprobs, new_values, _ = token_actor_critic_outputs(actor_state, batch)

print('token_ids:', batch.token_ids.tolist())
print('token_mask:', batch.token_mask.tolist())
print('policy_mask:', batch.policy_mask.tolist(), '(safe PPO would exclude fallback tokens)')
print('returns:', returns.tolist())
print('real Qwen actor logprobs:', qwen_scores.token_logprobs.tolist())
print('real Qwen actor entropy:', qwen_scores.entropy.tolist())
print('real Qwen critic head values:', qwen_scores.values.tolist())
print('old trainable smoke logprobs:', old_new_logprobs.tolist())
print('old smoke critic values:', old_values.tolist())
print('new smoke critic values:', new_values.tolist())
print('full-token PPO loss:', float(metrics['loss']))
print({name: float(value) for name, value in metrics.items()})
print('learned tokens:', float(metrics['learned_tokens']))
print('actor changed:', bool(jnp.any(jnp.abs(new_new_logprobs - old_new_logprobs) > 1e-7)))


In [ ]:
profile_path = ROOT / 'artifacts' / 'profiles' / 'full-cycle-craftext-training.json'
save_profile(
    profile_path,
    profiler.events(),
    metadata={
        'commit': git_revision(),
        'config': str(config_path.relative_to(ROOT)),
        'model': actor.profile.model_id,
        'batch_size': BATCH_SIZE,
        'horizon': HORIZON,
        'max_new_tokens': MAX_NEW_TOKENS,
    },
)
print('profile:', profile_path)
profiler.summary()


## Что проверяет notebook

- `QwenTunixBackend` реально генерирует completions из local Qwen snapshot; offline/mock backend отсутствует.
- `TunixCausalLmActor.score_tokens()` реально пересчитывает actor logprobs, entropy и value-head critic values по сохранённым prompt/generated tokens.
- Replay сохраняется до обучения и содержит prompt, raw completion, action, token ids/logprobs, rewards, masks и fallback evidence.
- `full_token_ppo_update()` выполняет локальный trainable PPO smoke по всем generated tokens из `token_mask`; внутри используется `masked_token_ppo_loss`, padding исключён.
- `PhaseProfiler` сохраняет JSON evidence по фазам rollout/replay/trajectory/actor/ppo.
- Для масштабного обучения используйте hardware-gated Tunix/RLCluster path; этот notebook — проверяемый single-process full-cycle contour.
